# fastText LID-176 Specialist-Head Evaluation

This notebook evaluates the **final frozen-backbone fastText adaptation**:

- Base model: `models/benchmark/fastText/lid.176.bin`
- Specialist head: `models/finetuned/fastText_LID_176/specialist_head_final.bin`
- Router: send text to the specialist when Sinhala-script ratio is `>= 0.5`

This notebook **does not train or load** `fasttext_lid_176_finetuned.bin`.

Recommended order:
1. Load the base model and specialist head.
2. Verify on `val.csv` that Python reproduces the C++ validation result.
3. Evaluate the untouched final system on `test.csv`.
4. Evaluate CommonLID, FLORES+ and WiLI-2018.
5. Check preservation of old-language predictions.


In [11]:
import sys
import numpy as np
import scipy
import sklearn

print(sys.executable)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("sklearn:", sklearn.__version__)

d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Scripts\python.exe
NumPy: 1.26.4
SciPy: 1.11.4
sklearn: 1.4.2


In [12]:
import sys
import numpy as np

print("Python:", sys.executable)
print("NumPy:", np.__version__)

# NumPy 2.x is okay in this corrected notebook.
# We avoid the old fastText single-string predict() wrapper bug below
# by using batch prediction with a one-item list.


Python: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Scripts\python.exe
NumPy: 1.26.4


In [13]:
import os
import json
import glob
import struct
from pathlib import Path

import numpy as np
import pandas as pd
import fasttext

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)


In [14]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")

    from google.colab import drive
    drive.mount('/content/drive')

    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'

    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(
            f'git clone https://github.com/Maleesha-K/'
            f'Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git '
            f'{repo_path}'
        )

    os.chdir(repo_path + '/data_pipeline')

    print("Installing dependencies...")
    os.system('pip install -q numpy pandas scikit-learn fasttext')

    print("Setup complete!")


In [15]:
# ------------------------------------------------------------
# Robust project paths
# ------------------------------------------------------------
# This works whether VS Code starts the notebook from:
# - project root
# - data_pipeline/
# - scripts/06.finetune_models/
# ------------------------------------------------------------

def find_data_pipeline_root():
    cwd = Path.cwd().resolve()

    # Case 1: current folder or one of its parents is data_pipeline
    for p in [cwd, *cwd.parents]:
        if p.name == "data_pipeline":
            return p

    # Case 2: current folder or one of its parents contains data_pipeline/
    for p in [cwd, *cwd.parents]:
        candidate = p / "data_pipeline"
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the data_pipeline folder automatically. "
        "Open the notebook from inside the project repository."
    )


DATA_PIPELINE_ROOT = find_data_pipeline_root()

BASE_MODEL_PATH = DATA_PIPELINE_ROOT / "models/benchmark/fastText/lid.176.bin"

SPECIALIST_HEAD_PATH = (
    DATA_PIPELINE_ROOT
    / "models/finetuned/fastText_LID_176/specialist_head_final.bin"
)

BENCHMARK_DIR = DATA_PIPELINE_ROOT / "datasets/preprocessed"
RESULTS_DIR = DATA_PIPELINE_ROOT / "datasets/benchmark_results"

VAL_PATH = DATA_PIPELINE_ROOT / "test_dataset_folder/val.csv"
TEST_PATH = DATA_PIPELINE_ROOT / "test_dataset_folder/test.csv"

ROUTER_THRESHOLD = 0.5

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("data_pipeline root :", DATA_PIPELINE_ROOT)
print("Base model        :", BASE_MODEL_PATH, "->", BASE_MODEL_PATH.exists())
print("Specialist head   :", SPECIALIST_HEAD_PATH, "->", SPECIALIST_HEAD_PATH.exists())
print("Benchmark folder  :", BENCHMARK_DIR, "->", BENCHMARK_DIR.exists())
print("Validation file   :", VAL_PATH, "->", VAL_PATH.exists())
print("Test file         :", TEST_PATH, "->", TEST_PATH.exists())


data_pipeline root : D:\Projects\ML Projects\LangID - DSE project\data_pipeline
Base model        : D:\Projects\ML Projects\LangID - DSE project\data_pipeline\models\benchmark\fastText\lid.176.bin -> True
Specialist head   : D:\Projects\ML Projects\LangID - DSE project\data_pipeline\models\finetuned\fastText_LID_176\specialist_head_final.bin -> True
Benchmark folder  : D:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\preprocessed -> True
Validation file   : D:\Projects\ML Projects\LangID - DSE project\data_pipeline\test_dataset_folder\val.csv -> True
Test file         : D:\Projects\ML Projects\LangID - DSE project\data_pipeline\test_dataset_folder\test.csv -> True


## Load the frozen LID-176 model and the 3 × 16 specialist head

The custom specialist head was stored as a fastText `DenseMatrix`.

The standard fastText `DenseMatrix::save()` layout is:

1. number of rows as `int64`
2. number of columns as `int64`
3. matrix values as `float32`

For the selected specialist head we expect exactly **3 rows × 16 columns**.

If this cell reports a different shape or file format, stop and compare it with your C++ `saveSpecialistHead()` implementation before continuing.


In [16]:
# Load original frozen LID-176

if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Base model not found: {BASE_MODEL_PATH}"
    )

base_model = fasttext.load_model(str(BASE_MODEL_PATH))

print("Loaded LID-176")
print("Dimension:", base_model.get_dimension())
print("Number of original labels:", len(base_model.get_labels()))


Loaded LID-176
Dimension: 16
Number of original labels: 176


In [17]:
# Load specialist_head_final.bin
# Expected binary format:
# int64 rows, int64 cols, followed by float32 matrix values.

def load_dense_matrix_bin(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Specialist head not found: {path}"
        )

    with path.open("rb") as f:
        header = f.read(16)

        if len(header) != 16:
            raise ValueError(
                "Specialist file is too small to contain a DenseMatrix header."
            )

        rows, cols = struct.unpack("<qq", header)
        raw = f.read()

    expected_bytes = rows * cols * 4

    if len(raw) != expected_bytes:
        raise ValueError(
            f"Unexpected specialist-head file layout.\n"
            f"Header says shape = {rows} x {cols}, requiring "
            f"{expected_bytes} bytes of float32 values, "
            f"but found {len(raw)} bytes."
        )

    return np.frombuffer(raw, dtype="<f4").reshape(rows, cols).copy()


specialist_W = load_dense_matrix_bin(SPECIALIST_HEAD_PATH)

print("Specialist matrix shape:", specialist_W.shape)

assert specialist_W.shape[0] == 3, (
    f"Expected 3 specialist classes, got {specialist_W.shape[0]}"
)

assert specialist_W.shape[1] == base_model.get_dimension(), (
    f"Specialist dimension {specialist_W.shape[1]} does not match "
    f"LID-176 dimension {base_model.get_dimension()}"
)

print("Specialist head loaded successfully.")


Specialist matrix shape: (3, 16)
Specialist head loaded successfully.


## Prediction functions

Specialist class order used during training:

- `0` → Sinhala
- `1` → Pali
- `2` → Sanskrit

The router calculates the proportion of alphabetic characters that belong to the Sinhala Unicode block `U+0D80–U+0DFF`.

If that ratio is at least `0.5`, the specialist is used. Otherwise the untouched LID-176 classifier is used.


In [18]:
SPECIALIST_LABELS = ["sinhala", "pali", "sanskrit"]

# Map original LID-176 codes used in our benchmark to readable names.
BASE_LABEL_TO_NAME = {
    "si": "sinhala",
    "sa": "sanskrit_deva",
    "en": "english",
    "ta": "tamil",
    "hi": "hindi",
    "bn": "bengali",
    "ar": "arabic",
    "fr": "french",
    "de": "german",
}


def format_text(text):
    return str(text).replace("\n", " ").strip()


def sinhala_script_ratio(text):
    text = format_text(text)

    # Count only alphabetic Unicode characters.
    alphabetic_chars = [ch for ch in text if ch.isalpha()]

    if not alphabetic_chars:
        return 0.0

    sinhala_letters = sum(
        1
        for ch in alphabetic_chars
        if 0x0D80 <= ord(ch) <= 0x0DFF
    )

    return sinhala_letters / len(alphabetic_chars)


def softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum()


def predict_specialist(text):
    text = format_text(text)

    sentence_vector = np.asarray(
        base_model.get_sentence_vector(text),
        dtype=np.float32,
    )

    scores = specialist_W @ sentence_vector
    probabilities = softmax(scores)

    class_id = int(np.argmax(probabilities))

    return (
        SPECIALIST_LABELS[class_id],
        float(probabilities[class_id]),
    )


def predict_base(text):
    text = format_text(text)

    # IMPORTANT:
    # Use a ONE-ITEM LIST instead of a single string.
    # The old fastText Python single-string wrapper calls:
    # np.array(probs, copy=False), which fails with NumPy 2.x.
    # Batch prediction avoids that wrapper branch.
    all_labels, all_probs = base_model.predict([text], k=1)

    labels = all_labels[0]
    probs = all_probs[0]

    if len(labels) == 0:
        return "unknown", 0.0

    code = labels[0].replace("__label__", "")
    mapped = BASE_LABEL_TO_NAME.get(code, code)

    return mapped, float(probs[0])


def predict_routed(text, threshold=ROUTER_THRESHOLD):
    ratio = sinhala_script_ratio(text)

    if ratio >= threshold:
        pred, confidence = predict_specialist(text)
        route = "specialist"
    else:
        pred, confidence = predict_base(text)
        route = "base"

    return {
        "predicted_label": pred,
        "route": route,
        "sinhala_script_ratio": ratio,
        "confidence": confidence,
    }


# Quick sanity checks
examples = [
    "මෙය සිංහල වාක්‍යයකි",
    "This is an English sentence.",
    "125",
]

for text in examples:
    print(text, "->", predict_routed(text))


මෙය සිංහල වාක්‍යයකි -> {'predicted_label': 'sinhala', 'route': 'specialist', 'sinhala_script_ratio': 1.0, 'confidence': 0.8062068361043023}
This is an English sentence. -> {'predicted_label': 'english', 'route': 'base', 'sinhala_script_ratio': 0.0, 'confidence': 0.9804080724716187}
125 -> {'predicted_label': 'french', 'route': 'base', 'sinhala_script_ratio': 0.0, 'confidence': 0.24390187859535217}


## Label-normalization helpers

These reproduce the label mapping used in the old evaluation notebook, so the new system is compared on the same benchmark labels.


In [19]:
TARGET_LANGUAGES = ["sinhala", "pali", "sanskrit"]

ALL_BENCHMARK_LANGUAGES = [
    "sinhala",
    "pali",
    "sanskrit",
    "sanskrit_deva",
    "english",
    "tamil",
    "hindi",
    "bengali",
    "arabic",
    "french",
    "german",
]

LABEL_MAPPING_ALL = {
    "sin": "sinhala",
    "sin_Sinh": "sinhala",
    "sinhala": "sinhala",
    "Sinhala": "sinhala",
    "si": "sinhala",

    "pli": "pali",
    "pli_Sinh": "pali",
    "pli_Latn": "pali",
    "pali": "pali",
    "Pali": "pali",
    "pi": "pali",

    "san_Sinh": "sanskrit",
    "sanskrit": "sanskrit",
    "Sanskrit": "sanskrit",

    "san_Deva": "sanskrit_deva",
    "sa": "sanskrit_deva",

    "eng": "english",
    "eng_Latn": "english",
    "english": "english",
    "en": "english",

    "tam": "tamil",
    "tam_Taml": "tamil",
    "tamil": "tamil",
    "ta": "tamil",

    "hin": "hindi",
    "hin_Deva": "hindi",
    "hindi": "hindi",
    "hi": "hindi",

    "ben": "bengali",
    "ben_Beng": "bengali",
    "bengali": "bengali",
    "bn": "bengali",

    "arb": "arabic",
    "arb_Arab": "arabic",
    "arabic": "arabic",
    "ar": "arabic",

    "fra": "french",
    "fra_Latn": "french",
    "french": "french",
    "fr": "french",

    "deu": "german",
    "deu_Latn": "german",
    "german": "german",
    "de": "german",
}


def map_target_label(row):
    lbl = str(row.get("label", "")).strip()
    src = row.get("source")

    if lbl in ["sin", "sin_Sinh", "sinhala", "Sinhala", "si"]:
        return "sinhala"

    if lbl in ["pli", "pli_Sinh", "pali", "Pali", "pi"]:
        return "pali"

    if lbl in ["san_Sinh", "sanskrit", "Sanskrit"]:
        return "sanskrit"

    if lbl == "san" and src in ["DCS", "SansinNT", "SiDiaC-v2"]:
        return "sanskrit"

    return None


def map_all_label(row):
    lbl = str(row.get("label", "")).strip()
    src = row.get("source")

    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        return "sanskrit_deva"

    return LABEL_MAPPING_ALL.get(lbl)


def normalize_project_label(label):
    label = str(label).strip()

    mapping = {
        "Sinhala": "sinhala",
        "sinhala": "sinhala",
        "sin": "sinhala",
        "si": "sinhala",

        "Pali": "pali",
        "pali": "pali",
        "pli": "pali",
        "pi": "pali",

        "Sanskrit": "sanskrit",
        "sanskrit": "sanskrit",
        "san": "sanskrit",
        "san_Sinh": "sanskrit",
    }

    return mapping.get(label, label.lower())


## First: verify against `val.csv`

**Do this before running the final test set.**

Your earlier C++ routed validation result was approximately:

- Accuracy: `0.8159`
- Macro F1: `0.7903`
- Router coverage: `0.9930`

The Python implementation should be very close to those values.

If it is not, do **not** continue to final testing yet. The Python loading/router logic needs to be checked against the C++ implementation.


In [20]:
def evaluate_project_csv(csv_path, name):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        print(f"{name} file not found: {csv_path}")
        return None

    df = pd.read_csv(csv_path)

    required = {"text", "label"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{csv_path} is missing columns: {missing}"
        )

    results = df.copy()

    results["true_label"] = results["label"].apply(
        normalize_project_label
    )

    predictions = results["text"].apply(
        predict_routed
    )

    results["predicted_label"] = predictions.apply(
        lambda x: x["predicted_label"]
    )

    results["route"] = predictions.apply(
        lambda x: x["route"]
    )

    results["sinhala_script_ratio"] = predictions.apply(
        lambda x: x["sinhala_script_ratio"]
    )

    results["confidence"] = predictions.apply(
        lambda x: x["confidence"]
    )

    y_true = results["true_label"].tolist()
    y_pred = results["predicted_label"].tolist()

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=TARGET_LANGUAGES,
        average="macro",
        zero_division=0
    )

    specialist_count = (
        results["route"] == "specialist"
    ).sum()

    router_coverage = (
        specialist_count / len(results)
        if len(results)
        else 0.0
    )

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(f"Rows:             {len(results)}")
    print(f"Accuracy:         {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Macro F1:         {macro_f1:.4f} ({macro_f1*100:.2f}%)")
    print(f"Specialist rows:  {specialist_count}")
    print(f"Router coverage:  {router_coverage:.4f} ({router_coverage*100:.2f}%)")

    print("\nPer-language results:\n")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=TARGET_LANGUAGES,
            digits=4,
            zero_division=0
        )
    )

    # Convert any non-target prediction to "other"
    cm_pred = [
        pred if pred in TARGET_LANGUAGES else "other"
        for pred in y_pred
    ]

    cm_labels = TARGET_LANGUAGES + ["other"]

    cm = confusion_matrix(
        y_true,
        cm_pred,
        labels=cm_labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"true_{x}"
            for x in cm_labels
        ],
        columns=[
            f"pred_{x}"
            for x in cm_labels
        ]
    )

    print("\nConfusion matrix:")
    display(cm_df)

    return {
        "results": results,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "router_coverage": router_coverage,
        "confusion_matrix": cm_df,
    }


val_eval = evaluate_project_csv(
    VAL_PATH,
    "VALIDATION — FINAL ROUTED FASTTEXT SYSTEM"
)


VALIDATION — FINAL ROUTED FASTTEXT SYSTEM
Rows:             6986
Accuracy:         0.8159 (81.59%)
Macro F1:         0.7905 (79.05%)
Specialist rows:  6937
Router coverage:  0.9930 (99.30%)

Per-language results:

              precision    recall  f1-score   support

     sinhala     0.8444    0.7627    0.8015      2895
        pali     0.9100    0.9243    0.9171      2800
    sanskrit     0.6116    0.7002    0.6529      1291

   micro avg     0.8217    0.8159    0.8188      6986
   macro avg     0.7887    0.7957    0.7905      6986
weighted avg     0.8277    0.8159    0.8204      6986


Confusion matrix:


,pred_sinhala,pred_pali,pred_sanskrit,pred_other
true_sinhala,2208,169,470,48
true_pali,107,2588,104,1
true_sanskrit,300,87,904,0
true_other,0,0,0,0


## Final held-out `test.csv`

Only run this after the validation result confirms that the Python implementation matches the C++ implementation closely.

The full test set should be kept unchanged, including numeric/non-alphabetic samples.


In [21]:
test_eval = evaluate_project_csv(
    TEST_PATH,
    "FINAL HELD-OUT TEST — ROUTED FASTTEXT SYSTEM"
)

if test_eval is not None:
    test_output = (
        RESULTS_DIR
        / "fasttext_lid_176_specialist_final_test.csv"
    )

    test_eval["results"].to_csv(test_output, index=False)

    print(f"\nSaved final test predictions to: {test_output}")



FINAL HELD-OUT TEST — ROUTED FASTTEXT SYSTEM
Rows:             7047
Accuracy:         0.8179 (81.79%)
Macro F1:         0.7982 (79.82%)
Specialist rows:  6984
Router coverage:  0.9911 (99.11%)

Per-language results:

              precision    recall  f1-score   support

     sinhala     0.8301    0.7895    0.8093      2693
        pali     0.9230    0.8751    0.8984      3027
    sanskrit     0.6368    0.7453    0.6868      1327

   micro avg     0.8253    0.8179    0.8216      7047
   macro avg     0.7967    0.8033    0.7982      7047
weighted avg     0.8336    0.8179    0.8245      7047


Confusion matrix:


,pred_sinhala,pred_pali,pred_sanskrit,pred_other
true_sinhala,2126,148,357,62
true_pali,170,2649,207,1
true_sanskrit,265,73,989,0
true_other,0,0,0,0



Saved final test predictions to: D:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\benchmark_results\fasttext_lid_176_specialist_final_test.csv


## Benchmark datasets: CommonLID, FLORES+ and WiLI-2018

This section evaluates the **same `datasets/preprocessed/*.jsonl` files** used by the old notebook, but now predictions come from:

`router → specialist_head_final.bin OR untouched lid.176.bin`


In [22]:
def load_target_benchmark(file_path):
    records = []

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped = map_target_label(row)

            if mapped:
                row["target_label"] = mapped
                records.append(row)

    return pd.DataFrame(records)


def load_all_benchmark(file_path):
    records = []

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped = map_all_label(row)

            if mapped:
                row["target_label"] = mapped
                records.append(row)

    return pd.DataFrame(records)


benchmark_files = sorted(BENCHMARK_DIR.glob("*.jsonl"))

print("Benchmark files found:")
for path in benchmark_files:
    print(" -", path.name)

if not benchmark_files:
    print("WARNING: No .jsonl benchmark files were found.")


Benchmark files found:
 - commonlid.jsonl
 - flores_plus.jsonl
 - wili-2018.jsonl


### A. Target-language benchmark

Evaluates Sinhala, Pali and Sinhala-script Sanskrit only.


In [23]:
target_summary = []

for file_path in benchmark_files:
    dataset_name = Path(file_path).stem
    df_bench = load_target_benchmark(file_path)

    if df_bench.empty:
        print(f"No matching target-language rows in {dataset_name}.")
        continue

    predictions = df_bench["text"].apply(predict_routed)

    results = df_bench.copy()
    results["predicted_label"] = predictions.apply(lambda x: x["predicted_label"])
    results["route"] = predictions.apply(lambda x: x["route"])
    results["sinhala_script_ratio"] = predictions.apply(lambda x: x["sinhala_script_ratio"])
    results["confidence"] = predictions.apply(lambda x: x["confidence"])

    y_true = results["target_label"]
    y_pred = results["predicted_label"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=TARGET_LANGUAGES,
        average="macro",
        zero_division=0
    )

    router_coverage = (results["route"] == "specialist").mean()

    print("\n" + "=" * 70)
    print(f"TARGET BENCHMARK — {dataset_name}")
    print("=" * 70)
    print(f"Rows:            {len(results)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")
    print(f"Macro F1:        {macro_f1:.4f} ({macro_f1*100:.2f}%)")
    print(f"Router coverage: {router_coverage:.4f} ({router_coverage*100:.2f}%)")
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=TARGET_LANGUAGES,
        digits=4,
        zero_division=0
    ))

    out_csv = os.path.join(
        RESULTS_DIR,
        f"fasttext_lid_176_specialist_{dataset_name}.csv"
    )
    results.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    target_summary.append({
        "dataset": dataset_name,
        "rows": len(results),
        "accuracy": acc,
        "macro_f1": macro_f1,
        "router_coverage": router_coverage,
    })


target_summary_df = pd.DataFrame(target_summary)
display(target_summary_df)



TARGET BENCHMARK — commonlid
Rows:            7047
Accuracy:        0.8179 (81.79%)
Macro F1:        0.7982 (79.82%)
Router coverage: 0.9911 (99.11%)

              precision    recall  f1-score   support

     sinhala     0.8301    0.7895    0.8093      2693
        pali     0.9230    0.8751    0.8984      3027
    sanskrit     0.6368    0.7453    0.6868      1327

   micro avg     0.8253    0.8179    0.8216      7047
   macro avg     0.7967    0.8033    0.7982      7047
weighted avg     0.8336    0.8179    0.8245      7047

Saved: D:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\benchmark_results\fasttext_lid_176_specialist_commonlid.csv

TARGET BENCHMARK — flores_plus
Rows:            7047
Accuracy:        0.8179 (81.79%)
Macro F1:        0.7982 (79.82%)
Router coverage: 0.9911 (99.11%)

              precision    recall  f1-score   support

     sinhala     0.8301    0.7895    0.8093      2693
        pali     0.9230    0.8751    0.8984      3027
    sanskrit   

,dataset,rows,accuracy,macro_f1,router_coverage
0,commonlid,7047,0.817937,0.798172,0.99106
1,flores_plus,7047,0.817937,0.798172,0.99106
2,wili-2018,7047,0.817937,0.798172,0.99106


### B. Selected multilingual benchmark

Evaluates the target languages together with the same selected old languages used in the old notebook.


In [24]:
all_language_summary = []

for file_path in benchmark_files:
    dataset_name = Path(file_path).stem
    df_all = load_all_benchmark(file_path)

    if df_all.empty:
        print(f"No matching benchmark rows in {dataset_name}.")
        continue

    predictions = df_all["text"].apply(predict_routed)

    results = df_all.copy()
    results["predicted_label"] = predictions.apply(lambda x: x["predicted_label"])
    results["route"] = predictions.apply(lambda x: x["route"])
    results["sinhala_script_ratio"] = predictions.apply(lambda x: x["sinhala_script_ratio"])
    results["confidence"] = predictions.apply(lambda x: x["confidence"])

    y_true = results["target_label"]
    y_pred = results["predicted_label"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=ALL_BENCHMARK_LANGUAGES,
        average="macro",
        zero_division=0
    )

    print("\n" + "=" * 70)
    print(f"ALL-LANGUAGE BENCHMARK — {dataset_name}")
    print("=" * 70)
    print(f"Rows:     {len(results)}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"Macro F1: {macro_f1:.4f} ({macro_f1*100:.2f}%)")
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=ALL_BENCHMARK_LANGUAGES,
        digits=4,
        zero_division=0
    ))

    out_csv = os.path.join(
        RESULTS_DIR,
        f"fasttext_lid_176_specialist_all_langs_{dataset_name}.csv"
    )
    results.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    all_language_summary.append({
        "dataset": dataset_name,
        "rows": len(results),
        "accuracy": acc,
        "macro_f1": macro_f1,
    })


all_language_summary_df = pd.DataFrame(all_language_summary)
display(all_language_summary_df)



ALL-LANGUAGE BENCHMARK — commonlid
Rows:     77974
Accuracy: 0.9565 (95.65%)
Macro F1: 0.9173 (91.73%)

               precision    recall  f1-score   support

      sinhala     0.8298    0.7895    0.8091      2693
         pali     0.9230    0.8751    0.8984      3027
     sanskrit     0.6368    0.7453    0.6868      1327
sanskrit_deva     0.9824    0.8112    0.8886       895
      english     0.9848    0.9603    0.9724     27461
        tamil     0.9310    1.0000    0.9643        81
        hindi     0.9721    0.9684    0.9702      3666
      bengali     0.9984    0.9889    0.9936      1886
       arabic     0.9997    0.9898    0.9947     26152
       french     0.9413    0.9480    0.9447      3233
       german     0.9715    0.9629    0.9672      7553

    micro avg     0.9718    0.9565    0.9641     77974
    macro avg     0.9246    0.9127    0.9173     77974
 weighted avg     0.9727    0.9565    0.9643     77974

Saved: D:\Projects\ML Projects\LangID - DSE project\data_pipeline\d

,dataset,rows,accuracy,macro_f1
0,commonlid,77974,0.956485,0.917281
1,flores_plus,16155,0.853049,0.880121
2,wili-2018,14047,0.897202,0.838416


## Old-language preservation check

For non-target benchmark rows, compare:

- prediction from the untouched `lid.176.bin`
- prediction from the final routed system

If the router sends none of these old-language rows to the specialist, the final predictions should be identical to the original model.


In [25]:
preservation_summary = []

for file_path in benchmark_files:
    dataset_name = Path(file_path).stem
    df_all = load_all_benchmark(file_path)

    if df_all.empty:
        continue

    old_df = df_all[
        ~df_all["target_label"].isin(["sinhala", "pali", "sanskrit"])
    ].copy()

    if old_df.empty:
        continue

    base_preds = old_df["text"].apply(lambda x: predict_base(x)[0])
    routed_outputs = old_df["text"].apply(predict_routed)

    routed_preds = routed_outputs.apply(lambda x: x["predicted_label"])
    routes = routed_outputs.apply(lambda x: x["route"])

    sent_to_specialist = int((routes == "specialist").sum())
    changed = int((base_preds.values != routed_preds.values).sum())
    preservation_rate = 1.0 - changed / len(old_df)

    print("\n" + "=" * 70)
    print(f"OLD-LANGUAGE PRESERVATION — {dataset_name}")
    print("=" * 70)
    print(f"Old-language rows:       {len(old_df)}")
    print(f"Sent to specialist:      {sent_to_specialist}")
    print(f"Predictions changed:     {changed}")
    print(f"Prediction preservation: {preservation_rate:.4f} ({preservation_rate*100:.2f}%)")

    preservation_summary.append({
        "dataset": dataset_name,
        "old_rows": len(old_df),
        "sent_to_specialist": sent_to_specialist,
        "predictions_changed": changed,
        "preservation_rate": preservation_rate,
    })


preservation_summary_df = pd.DataFrame(preservation_summary)
display(preservation_summary_df)



OLD-LANGUAGE PRESERVATION — commonlid
Old-language rows:       70927
Sent to specialist:      0
Predictions changed:     0
Prediction preservation: 1.0000 (100.00%)

OLD-LANGUAGE PRESERVATION — flores_plus
Old-language rows:       9108
Sent to specialist:      0
Predictions changed:     0
Prediction preservation: 1.0000 (100.00%)

OLD-LANGUAGE PRESERVATION — wili-2018
Old-language rows:       7000
Sent to specialist:      0
Predictions changed:     0
Prediction preservation: 1.0000 (100.00%)


,dataset,old_rows,sent_to_specialist,predictions_changed,preservation_rate
0,commonlid,70927,0,0,1.0
1,flores_plus,9108,0,0,1.0
2,wili-2018,7000,0,0,1.0


## Final summary

For the mentor/report, record:

- final test accuracy
- final test Macro F1
- Sinhala precision / recall / F1
- Pali precision / recall / F1
- Sanskrit precision / recall / F1
- confusion matrix
- router coverage
- old-language preservation
- exact configuration:
  - frozen `lid.176.bin`
  - class-weighted `specialist_head_final.bin`
  - router threshold `0.5`

Do not tune the model after seeing the held-out final test result.


In [26]:
print("\n" + "=" * 70)
print("FASTTEXT SPECIALIST-HEAD EVALUATION COMPLETE")
print("=" * 70)
print("Base model:      ", BASE_MODEL_PATH)
print("Specialist head: ", SPECIALIST_HEAD_PATH)
print("Router threshold:", ROUTER_THRESHOLD)
print("Results folder:  ", RESULTS_DIR)
print("=" * 70)



FASTTEXT SPECIALIST-HEAD EVALUATION COMPLETE
Base model:       D:\Projects\ML Projects\LangID - DSE project\data_pipeline\models\benchmark\fastText\lid.176.bin
Specialist head:  D:\Projects\ML Projects\LangID - DSE project\data_pipeline\models\finetuned\fastText_LID_176\specialist_head_final.bin
Router threshold: 0.5
Results folder:   D:\Projects\ML Projects\LangID - DSE project\data_pipeline\datasets\benchmark_results
